In [1]:
# Cell 1: Installation
!pip install diffusers transformers accelerate torch torchvision Pillow --quiet

^C


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\jc301368\\AppData\\Roaming\\Python\\Python313\\site-packages\\torchvision\\tv_tensors\\__init__.py'
Check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [ ]:
# Cell 2: Imports and device setup
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import os
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'Using device: {device}')

In [ ]:
# Cell 3: Model loading
# Using runwayml/stable-diffusion-v1-5 -- free and widely available on Hugging Face.
# If you get a 401 error, create a free account at https://huggingface.co,
# generate a token at https://huggingface.co/settings/tokens, then run:
#   from huggingface_hub import login; login(token='YOUR_TOKEN_HERE')

MODEL_ID = 'runwayml/stable-diffusion-v1-5'

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to(device)

if device == 'cpu':
    pipe.enable_attention_slicing()

print('Model loaded successfully.')

In [ ]:
# Cell 4: Prompt definition
BASE_STYLE = (
    'semi-realistic animated film style, cinematic lighting, high detail, '
    '16:9 wide composition, no text, no watermarks, photorealistic rendering, '
    'small cute round white exploration robot with glowing blue sensors, '
    'overgrown abandoned city, crumbling concrete buildings covered in vines'
)

NEGATIVE_PROMPT = (
    'text, watermark, signature, blurry, low quality, deformed, ugly, '
    'cartoon, anime, extra limbs, multiple robots'
)

scene_prompts = [
    # Scene 1: Establishing shot
    (
        'Scene establishing shot: small cute round white exploration robot enters an overgrown abandoned city at sunset, '
        'warm golden hour light, long shadows, ruins of skyscrapers covered in ivy and moss, dramatic sky with orange and purple clouds, '
        + BASE_STYLE
    ),
    # Scene 2: Investigation
    (
        'Scene investigation: small cute round white exploration robot scanning surroundings with a bright blue light beam, '
        'blue scanning ray illuminates dusty air and rubble, close-up cinematic angle, mist rising from cracked ground, '
        'abandoned city street at dusk, ' + BASE_STYLE
    ),
    # Scene 3: Discovery
    (
        'Scene discovery: small cute round white exploration robot discovers a glowing bioluminescent plant growing through cracked concrete, '
        'plant emits soft cyan and green light, robot stares in awe, dark ruined environment, dramatic contrast between darkness and plant glow, '
        + BASE_STYLE
    ),
    # Scene 4: Interaction
    (
        'Scene interaction: small cute round white exploration robot gently touches a glowing bioluminescent plant with its robotic arm, '
        'plant glow intensifies brilliantly illuminating surrounding ruins, magical light explosion of cyan and white, '
        'emotional cinematic moment, ruins bathed in warm glow, ' + BASE_STYLE
    ),
]

print('Prompts defined for 4 scenes.')
for i, p in enumerate(scene_prompts, 1):
    print(f'Scene {i}: {p[:100]}...')

In [ ]:
# Cell 5: Image generation loop
SEED = 42
NUM_INFERENCE_STEPS = 30  # Reduce to 20 for faster CPU runs
GUIDANCE_SCALE = 7.5
WIDTH, HEIGHT = 768, 432  # 16:9 aspect ratio

generated_images = []

for i, prompt in enumerate(scene_prompts, 1):
    print(f'Generating scene {i}/4...')
    generator = torch.Generator(device=device).manual_seed(SEED)

    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )

    img = result.images[0]
    generated_images.append(img)
    print(f'Scene {i} generated.')

print('All 4 scenes generated successfully!')

In [ ]:
# Cell 6: Save outputs
output_dir = os.getcwd()
filenames = ['scene_1.png', 'scene_2.png', 'scene_3.png', 'scene_4.png']

for fname, img in zip(filenames, generated_images):
    path = os.path.join(output_dir, fname)
    img.save(path)
    print(f'Saved: {path}')

print('All scenes saved to disk.')

In [ ]:
# Cell 7: Quick preview
scene_titles = [
    'Scene 1 - Establishing Shot',
    'Scene 2 - Investigation',
    'Scene 3 - Discovery',
    'Scene 4 - Interaction',
]

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
axes = axes.flatten()

for ax, img, title in zip(axes, generated_images, scene_titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.axis('off')

plt.suptitle('Storyboard: Robot Discovers a Glowing Plant', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('storyboard_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Preview saved as storyboard_preview.png')